# Competition Push — IndabaX Zimbabwe 2026 Loan Default

**Goal**: Maximize public AUC from ~0.670 toward 0.683+

| Step | What | Est. Time |
|------|------|-----------|
| 1 | Enhanced features + encoding | 2 min |
| 2 | Optuna LGBM (40 trials, 3-fold) | ~20 min |
| 3 | Optuna XGB (30 trials, 3-fold) | ~15 min |
| 4 | Train 7 diverse models (5-fold) | ~30 min |
| 5 | Ensemble optimization + submission | 2 min |

In [ ]:
import os, sys, warnings
warnings.filterwarnings('ignore')

project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir(project_root)
print(f'Working dir: {os.getcwd()}')

!{sys.executable} -m pip install -q lightgbm xgboost catboost shap scipy scikit-learn pandas numpy optuna pydantic pyyaml pyarrow
!{sys.executable} -m pip install -q -e .

import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from sklearn.linear_model import Ridge
from sklearn.ensemble import ExtraTreesClassifier
from lightgbm import LGBMClassifier, early_stopping
import xgboost as xgb
from catboost import CatBoostClassifier, Pool
from scipy.stats import rankdata
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

from src.utils.seeding import seed_everything
from src.data.loader import (
    load_train, load_test, load_sample_submission,
    TARGET_COL, ID_COL, CATEGORICAL_COLS,
)
from src.features.base import BaseFeatureEngineer
from src.features.encoders.ordinal import OrdinalEncoder
from src.features.encoders.target import KFoldTargetEncoder
from src.features.encoders.woe import WOEEncoder
from src.features.encoders.group_stats import GroupStatsEncoder

seed_everything(42)
SEED = 42
N_FOLDS = 5
N_TUNE_FOLDS = 3

Working dir: C:\Users\Tinevimbo\Desktop\deep_learning_indabaX


In [ ]:
# ── Data + Enhanced Feature Engineering ──────────────────────────────
train_raw = load_train()
test_raw = load_test()
sample_sub = load_sample_submission()

fe = BaseFeatureEngineer()
y = train_raw[TARGET_COL].copy()
train_fe = fe.fit_transform(train_raw)
test_fe = fe.transform(test_raw)

# ── Extra features not in BaseFeatureEngineer ────────────────────────
def _add_competition_features(train_df, test_df):
    """Extra features: frequency encoding, urban flag, rate interactions."""
    cat_cols = [c for c in train_df.columns
                if train_df[c].dtype.name in ('category', 'object')]

    # Frequency encoding — compute from TRAIN, apply to both
    for col in cat_cols:
        freq_map = (
            train_df[col].astype(str)
            .value_counts(normalize=True)
            .to_dict()
        )
        for df in [train_df, test_df]:
            df[f'{col}_freq'] = (
                df[col].astype(str).map(freq_map)
                .fillna(0).astype(np.float32)
            )

    for df in [train_df, test_df]:
        # Zimbabwe urban provinces
        if 'province' in df.columns:
            df['is_urban'] = (
                df['province'].astype(str)
                .isin(['Harare', 'Bulawayo'])
                .astype(np.float32)
            )

        rate = df.get('annual_rate_pct')
        amount = df.get('amount_usd')
        income = df.get('monthly_income_usd')
        term = df.get('term_months')
        age = df.get('client_age_at_approval')

        # Rate × amount interaction — total interest burden
        if rate is not None and amount is not None:
            df['interest_burden'] = (
                rate * amount / 100
            ).astype(np.float32)

        # Rate relative to income
        if rate is not None and income is not None:
            df['rate_income_ratio'] = (
                rate / income.clip(lower=1)
            ).astype(np.float32)

        # Total cost relative to income (months of income needed)
        if amount is not None and rate is not None and term is not None and income is not None:
            total_cost = amount * (1 + rate / 100 * term.clip(lower=1) / 12)
            df['months_to_repay'] = (
                total_cost / income.clip(lower=1)
            ).astype(np.float32)

        # Age-based flags
        if age is not None:
            df['age_young'] = (age < 25).astype(np.float32)
            df['age_senior'] = (age > 55).astype(np.float32)
            if income is not None:
                df['age_income_ratio'] = (
                    age / income.clip(lower=1)
                ).astype(np.float32)

        # MFI × collateral interaction
        if 'is_mfi_loan' in df.columns and 'has_collateral' in df.columns:
            df['mfi_no_collateral'] = (
                (df['is_mfi_loan'] == 1) & (df['has_collateral'] == 0)
            ).astype(np.float32)

    return train_df, test_df

train_fe, test_fe = _add_competition_features(train_fe, test_fe)
yv = y.values

print(f'Train: {train_fe.shape}, Test: {test_fe.shape}')
print(f'Target rate: {yv.mean():.4f}')

Train: (38932, 73), Test: (12977, 72)
Target rate: 0.2412


In [ ]:
# ── Build encoding variants + helpers ────────────────────────────────

def get_Xy(df, exclude_cols=None):
    """Extract numeric feature matrix."""
    exclude = {ID_COL, TARGET_COL} | (exclude_cols or set())
    feat_cols = [c for c in df.columns
                 if c not in exclude and df[c].dtype.kind in ('i', 'f', 'u')]
    X = df[feat_cols].values.astype(np.float32)
    X = np.nan_to_num(X, nan=0.0)
    return X, feat_cols


def encode_v3_fold(tr_df, val_df, y_tr):
    """Encode one CV fold: target + WOE (low-card) + group stats."""
    cat_cols = [c for c in tr_df.columns
                if tr_df[c].dtype.name in ('category', 'object')]
    woe_cols = [c for c in cat_cols if tr_df[c].nunique() <= 30]

    te = KFoldTargetEncoder(cat_cols=cat_cols, smoothing=10.0)
    te.fit(tr_df, y_tr)
    tr_enc = te.transform(tr_df)
    val_enc = te.transform(val_df)

    if woe_cols:
        woe = WOEEncoder(cat_cols=woe_cols)
        woe.fit(tr_df, y_tr)
        tr_w = woe.transform(tr_df[woe_cols].copy())
        val_w = woe.transform(val_df[woe_cols].copy())
        for c in woe_cols:
            tr_enc[f'{c}_woe'] = tr_w[c].values
            val_enc[f'{c}_woe'] = val_w[c].values

    gs_cols = [c for c in ['province', 'employment_sector']
               if c in tr_df.columns]
    if gs_cols:
        gs = GroupStatsEncoder(group_cols=gs_cols)
        gs.fit(tr_df, y_tr)
        tr_g = gs.transform(tr_df[gs_cols].copy())
        val_g = gs.transform(val_df[gs_cols].copy())
        for c in tr_g.columns:
            if c not in gs_cols:
                tr_enc[c] = tr_g[c].values
                val_enc[c] = val_g[c].values
    return tr_enc, val_enc


# V2: ordinal (safe, fast, used for Optuna tuning)
ord_enc = OrdinalEncoder()
ord_enc.fit(train_fe)
tr_v2 = ord_enc.transform(train_fe)
te_v2 = ord_enc.transform(test_fe)

X_v2, feat_v2 = get_Xy(tr_v2)
Xt_v2, _ = get_Xy(te_v2)

# CatBoost-specific: keep original categoricals as strings
cat_col_names = [c for c in train_fe.columns
                 if train_fe[c].dtype.name in ('category', 'object')]

def _catboost_frame(df, feat_cols):
    out = df[feat_cols].copy()
    for c in out.columns:
        if out[c].dtype.name in ('category', 'object'):
            out[c] = out[c].astype(str).fillna('__NA__')
        else:
            out[c] = out[c].astype(np.float32).fillna(0)
    return out

cb_feat_cols = [c for c in train_fe.columns
                if c not in {ID_COL, TARGET_COL}
                and (train_fe[c].dtype.kind in ('i', 'f', 'u')
                     or train_fe[c].dtype.name in ('category', 'object'))]
X_cb_train = _catboost_frame(train_fe, cb_feat_cols)
X_cb_test = _catboost_frame(test_fe, cb_feat_cols)
cb_cat_idx = [i for i, c in enumerate(cb_feat_cols) if c in cat_col_names]

print(f'V2 features:       {X_v2.shape[1]}')
print(f'CatBoost features: {len(cb_feat_cols)} ({len(cb_cat_idx)} categorical)')
print('V3 encoding will happen inside each CV fold (leak-safe)')

V2 features:       71
CatBoost features: 71 (12 categorical)
V3 encoding will happen inside each CV fold (leak-safe)


## Optuna Hyperparameter Tuning

Tuning on **v2 (ordinal)** features with 3-fold CV for speed.
Best params will be used for all encoding variants in the final models.

In [ ]:
# ── Optuna LGBM tuning ───────────────────────────────────────────────
def tune_lgbm(X, yv, n_trials=40, n_folds=N_TUNE_FOLDS):
    skf = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=SEED)

    def objective(trial):
        p = {
            'n_estimators': 5000,
            'num_leaves': trial.suggest_int('num_leaves', 15, 127),
            'learning_rate': trial.suggest_float('lr', 0.005, 0.1, log=True),
            'min_child_samples': trial.suggest_int('min_child', 5, 100),
            'subsample': trial.suggest_float('subsample', 0.5, 1.0),
            'colsample_bytree': trial.suggest_float('colsample', 0.3, 1.0),
            'reg_alpha': trial.suggest_float('alpha', 1e-8, 10.0, log=True),
            'reg_lambda': trial.suggest_float('lambda', 1e-8, 10.0, log=True),
            'min_split_gain': trial.suggest_float('min_gain', 0.0, 1.0),
            'max_depth': trial.suggest_int('max_depth', -1, 12),
            'is_unbalance': True,
            'random_state': SEED, 'verbose': -1, 'metric': 'auc',
        }
        oof = np.zeros(len(X))
        for tr_i, va_i in skf.split(X, yv):
            m = LGBMClassifier(**p)
            m.fit(X[tr_i], yv[tr_i],
                  eval_set=[(X[va_i], yv[va_i])],
                  callbacks=[early_stopping(80, verbose=False)])
            oof[va_i] = m.predict_proba(X[va_i])[:, 1]
        return roc_auc_score(yv, oof)

    study = optuna.create_study(
        direction='maximize',
        sampler=optuna.samplers.TPESampler(seed=SEED),
    )
    study.optimize(objective, n_trials=n_trials, show_progress_bar=True)

    bp = study.best_params
    clean = {
        'num_leaves': bp['num_leaves'],
        'learning_rate': bp['lr'],
        'min_child_samples': bp['min_child'],
        'subsample': bp['subsample'],
        'colsample_bytree': bp['colsample'],
        'reg_alpha': bp['alpha'],
        'reg_lambda': bp['lambda'],
        'min_split_gain': bp['min_gain'],
        'max_depth': bp['max_depth'],
    }
    print(f'\nBest LGBM AUC: {study.best_value:.6f}')
    print(f'Best params: {clean}')
    return clean

print('Tuning LGBM (40 trials, 3-fold)...')
best_lgbm = tune_lgbm(X_v2, yv, n_trials=40)

Tuning LGBM (40 trials, 3-fold)...


  0%|          | 0/40 [00:00<?, ?it/s]


Best LGBM AUC: 0.688067
Best params: {'num_leaves': 26, 'learning_rate': 0.025384038107920254, 'min_child_samples': 31, 'subsample': 0.5980228583463162, 'colsample_bytree': 0.30168976786563795, 'reg_alpha': 1.1659022473424805, 'reg_lambda': 4.108409897029185e-05, 'min_split_gain': 0.8213885693504167, 'max_depth': 2}


In [ ]:
# ── Optuna XGBoost tuning ────────────────────────────────────────────
def tune_xgb(X, yv, n_trials=30, n_folds=N_TUNE_FOLDS):
    skf = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=SEED)
    sw = float((yv == 0).sum() / (yv == 1).sum())

    def objective(trial):
        p = {
            'n_estimators': 5000,
            'max_depth': trial.suggest_int('max_depth', 3, 10),
            'learning_rate': trial.suggest_float('lr', 0.005, 0.1, log=True),
            'min_child_weight': trial.suggest_int('min_child_weight', 1, 50),
            'subsample': trial.suggest_float('subsample', 0.5, 1.0),
            'colsample_bytree': trial.suggest_float('colsample', 0.3, 1.0),
            'reg_alpha': trial.suggest_float('alpha', 1e-8, 10.0, log=True),
            'reg_lambda': trial.suggest_float('lambda', 1e-8, 10.0, log=True),
            'gamma': trial.suggest_float('gamma', 0.0, 5.0),
            'scale_pos_weight': sw,
            'tree_method': 'hist', 'eval_metric': 'auc',
            'random_state': SEED, 'verbosity': 0,
            'early_stopping_rounds': 80,
        }
        oof = np.zeros(len(X))
        for tr_i, va_i in skf.split(X, yv):
            m = xgb.XGBClassifier(**p)
            m.fit(X[tr_i], yv[tr_i],
                  eval_set=[(X[va_i], yv[va_i])], verbose=False)
            oof[va_i] = m.predict_proba(X[va_i])[:, 1]
        return roc_auc_score(yv, oof)

    study = optuna.create_study(
        direction='maximize',
        sampler=optuna.samplers.TPESampler(seed=SEED),
    )
    study.optimize(objective, n_trials=n_trials, show_progress_bar=True)

    bp = study.best_params
    clean = {
        'max_depth': bp['max_depth'],
        'learning_rate': bp['lr'],
        'min_child_weight': bp['min_child_weight'],
        'subsample': bp['subsample'],
        'colsample_bytree': bp['colsample'],
        'reg_alpha': bp['alpha'],
        'reg_lambda': bp['lambda'],
        'gamma': bp['gamma'],
    }
    print(f'\nBest XGB AUC: {study.best_value:.6f}')
    print(f'Best params: {clean}')
    return clean

print('Tuning XGBoost (30 trials, 3-fold)...')
best_xgb = tune_xgb(X_v2, yv, n_trials=30)

Tuning XGBoost (30 trials, 3-fold)...


  0%|          | 0/30 [00:00<?, ?it/s]


Best XGB AUC: 0.688153
Best params: {'max_depth': 3, 'learning_rate': 0.014839870409360698, 'min_child_weight': 18, 'subsample': 0.8035776825921466, 'colsample_bytree': 0.541834079733241, 'reg_alpha': 3.895620084904e-06, 'reg_lambda': 0.013050374662615487, 'gamma': 2.5934646974174678}


## Train All Models (5-fold CV)

7 diverse models for a strong ensemble:
1. LGBM on v2 (ordinal) — tuned params
2. LGBM on v3 (target+WOE, within-fold) — tuned params
3. LGBM diverse (different seed + high regularization)
4. XGBoost on v2 — tuned params
5. XGBoost on v3 (within-fold)
6. CatBoost with native categoricals
7. ExtraTrees for decorrelation

In [ ]:
# ── Train all models and collect OOF + test predictions ──────────────
skf5 = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
sw = float((yv == 0).sum() / (yv == 1).sum())

oofs = {}
tests = {}

# --- Shared param dicts ---
lgbm_base = {
    **best_lgbm, 'n_estimators': 5000,
    'is_unbalance': True, 'random_state': SEED,
    'verbose': -1, 'metric': 'auc',
}
xgb_base = {
    **best_xgb, 'n_estimators': 5000,
    'scale_pos_weight': sw, 'tree_method': 'hist',
    'eval_metric': 'auc', 'random_state': SEED,
    'verbosity': 0, 'early_stopping_rounds': 200,
}


def _train_cv(name, X_tr, X_te, yv, model_fn, skf):
    """Generic 5-fold CV: returns (oof, test_avg)."""
    oof = np.zeros(len(yv))
    test_avg = np.zeros(X_te.shape[0])
    for fold, (ti, vi) in enumerate(skf.split(X_tr, yv)):
        m = model_fn()
        m.fit(X_tr[ti], yv[ti],
              eval_set=[(X_tr[vi], yv[vi])],
              callbacks=[early_stopping(200, verbose=False)])
        oof[vi] = m.predict_proba(X_tr[vi])[:, 1]
        test_avg += m.predict_proba(X_te)[:, 1] / N_FOLDS
    auc = roc_auc_score(yv, oof)
    print(f'  [{name:25s}] OOF AUC = {auc:.6f}')
    return oof, test_avg


def _train_cv_xgb(name, X_tr, X_te, yv, params, skf):
    """XGBoost CV (no callbacks kwarg)."""
    oof = np.zeros(len(yv))
    test_avg = np.zeros(X_te.shape[0])
    for fold, (ti, vi) in enumerate(skf.split(X_tr, yv)):
        m = xgb.XGBClassifier(**params)
        m.fit(X_tr[ti], yv[ti],
              eval_set=[(X_tr[vi], yv[vi])], verbose=False)
        oof[vi] = m.predict_proba(X_tr[vi])[:, 1]
        test_avg += m.predict_proba(X_te)[:, 1] / N_FOLDS
    auc = roc_auc_score(yv, oof)
    print(f'  [{name:25s}] OOF AUC = {auc:.6f}')
    return oof, test_avg


# 1) LGBM v2
print('=== LGBM v2 (tuned) ===')
oofs['lgbm_v2'], tests['lgbm_v2'] = _train_cv(
    'lgbm_v2', X_v2, Xt_v2, yv,
    lambda: LGBMClassifier(**lgbm_base), skf5,
)

# 2) LGBM v3 (within-fold encoding)
print('=== LGBM v3 (tuned, within-fold) ===')
oof_v3 = np.zeros(len(yv))
for fold, (ti, vi) in enumerate(skf5.split(np.zeros(len(yv)), yv)):
    tr_enc, val_enc = encode_v3_fold(
        train_fe.iloc[ti], train_fe.iloc[vi], y.iloc[ti])
    Xtr, _ = get_Xy(tr_enc)
    Xva, _ = get_Xy(val_enc)
    m = LGBMClassifier(**lgbm_base)
    m.fit(Xtr, yv[ti], eval_set=[(Xva, yv[vi])],
          callbacks=[early_stopping(200, verbose=False)])
    oof_v3[vi] = m.predict_proba(Xva)[:, 1]
# Test encoding: fit on ALL train
tr_v3f, te_v3f = encode_v3_fold(train_fe, test_fe, y)
Xv3f, _ = get_Xy(tr_v3f)
Xtv3f, _ = get_Xy(te_v3f)
test_v3 = np.zeros(len(Xtv3f))
for fold, (ti, vi) in enumerate(skf5.split(Xv3f, yv)):
    m = LGBMClassifier(**lgbm_base)
    m.fit(Xv3f[ti], yv[ti], eval_set=[(Xv3f[vi], yv[vi])],
          callbacks=[early_stopping(200, verbose=False)])
    test_v3 += m.predict_proba(Xtv3f)[:, 1] / N_FOLDS
oofs['lgbm_v3'] = oof_v3
tests['lgbm_v3'] = test_v3
print(f'  [{"lgbm_v3":25s}] OOF AUC = {roc_auc_score(yv, oof_v3):.6f}')

# 3) LGBM diverse (high regularization, different seed)
print('=== LGBM diverse ===')
lgbm_div = {
    'n_estimators': 5000, 'num_leaves': 127, 'learning_rate': 0.005,
    'min_child_samples': 50, 'subsample': 0.6, 'colsample_bytree': 0.4,
    'reg_alpha': 5.0, 'reg_lambda': 5.0, 'min_split_gain': 0.5,
    'is_unbalance': True, 'random_state': SEED + 1,
    'verbose': -1, 'metric': 'auc',
}
oofs['lgbm_div'], tests['lgbm_div'] = _train_cv(
    'lgbm_diverse', X_v2, Xt_v2, yv,
    lambda: LGBMClassifier(**lgbm_div), skf5,
)

# 4) XGB v2
print('=== XGB v2 (tuned) ===')
oofs['xgb_v2'], tests['xgb_v2'] = _train_cv_xgb(
    'xgb_v2', X_v2, Xt_v2, yv, xgb_base, skf5,
)

# 5) XGB v3 (within-fold encoding)
print('=== XGB v3 (tuned, within-fold) ===')
oof_xv3 = np.zeros(len(yv))
for fold, (ti, vi) in enumerate(skf5.split(np.zeros(len(yv)), yv)):
    tr_enc, val_enc = encode_v3_fold(
        train_fe.iloc[ti], train_fe.iloc[vi], y.iloc[ti])
    Xtr, _ = get_Xy(tr_enc)
    Xva, _ = get_Xy(val_enc)
    m = xgb.XGBClassifier(**xgb_base)
    m.fit(Xtr, yv[ti], eval_set=[(Xva, yv[vi])], verbose=False)
    oof_xv3[vi] = m.predict_proba(Xva)[:, 1]
test_xv3 = np.zeros(len(Xtv3f))
for fold, (ti, vi) in enumerate(skf5.split(Xv3f, yv)):
    m = xgb.XGBClassifier(**xgb_base)
    m.fit(Xv3f[ti], yv[ti], eval_set=[(Xv3f[vi], yv[vi])], verbose=False)
    test_xv3 += m.predict_proba(Xtv3f)[:, 1] / N_FOLDS
oofs['xgb_v3'] = oof_xv3
tests['xgb_v3'] = test_xv3
print(f'  [{"xgb_v3":25s}] OOF AUC = {roc_auc_score(yv, oof_xv3):.6f}')

# 6) CatBoost (native categoricals — leak-free by design)
print('=== CatBoost ===')
oof_cb = np.zeros(len(yv))
test_cb = np.zeros(len(X_cb_test))
for fold, (ti, vi) in enumerate(skf5.split(np.zeros(len(yv)), yv)):
    pool_tr = Pool(X_cb_train.iloc[ti], yv[ti], cat_features=cb_cat_idx)
    pool_va = Pool(X_cb_train.iloc[vi], yv[vi], cat_features=cb_cat_idx)
    m = CatBoostClassifier(
        iterations=3000, depth=6, learning_rate=0.05,
        l2_leaf_reg=3, eval_metric='AUC',
        random_seed=SEED, verbose=0,
        early_stopping_rounds=200,
    )
    m.fit(pool_tr, eval_set=pool_va)
    oof_cb[vi] = m.predict_proba(X_cb_train.iloc[vi])[:, 1]
    test_cb += m.predict_proba(X_cb_test)[:, 1] / N_FOLDS
oofs['catboost'] = oof_cb
tests['catboost'] = test_cb
print(f'  [{"catboost":25s}] OOF AUC = {roc_auc_score(yv, oof_cb):.6f}')

# 7) ExtraTrees (fast, decorrelated)
print('=== ExtraTrees ===')
oof_et = np.zeros(len(yv))
test_et = np.zeros(len(Xt_v2))
for fold, (ti, vi) in enumerate(skf5.split(X_v2, yv)):
    m = ExtraTreesClassifier(
        n_estimators=500, max_depth=None, min_samples_leaf=10,
        class_weight='balanced', random_state=SEED, n_jobs=-1,
    )
    m.fit(X_v2[ti], yv[ti])
    oof_et[vi] = m.predict_proba(X_v2[vi])[:, 1]
    test_et += m.predict_proba(Xt_v2)[:, 1] / N_FOLDS
oofs['et'] = oof_et
tests['et'] = test_et
print(f'  [{"extra_trees":25s}] OOF AUC = {roc_auc_score(yv, oof_et):.6f}')

print('\n=== All models done ===')
for name, oof in oofs.items():
    print(f'  {name:20s} AUC = {roc_auc_score(yv, oof):.6f}')

=== LGBM v2 (tuned) ===
  [lgbm_v2                  ] OOF AUC = 0.689141
=== LGBM v3 (tuned, within-fold) ===
  [lgbm_v3                  ] OOF AUC = 0.688054
=== LGBM diverse ===
  [lgbm_diverse             ] OOF AUC = 0.683833
=== XGB v2 (tuned) ===
  [xgb_v2                   ] OOF AUC = 0.688630
=== XGB v3 (tuned, within-fold) ===
  [xgb_v3                   ] OOF AUC = 0.687501
=== CatBoost ===
  [catboost                 ] OOF AUC = 0.690080
=== ExtraTrees ===
  [extra_trees              ] OOF AUC = 0.675045

=== All models done ===
  lgbm_v2              AUC = 0.689141
  lgbm_v3              AUC = 0.688054
  lgbm_div             AUC = 0.683833
  xgb_v2               AUC = 0.688630
  xgb_v3               AUC = 0.687501
  catboost             AUC = 0.690080
  et                   AUC = 0.675045


## Ensemble Optimization + Submission

Two ensemble strategies:
1. **Optuna blend** — optimize rank-average weights (200 trials, instant)
2. **Ridge stacking** — meta-learner on OOF predictions

Final = rank average of both strategies

In [ ]:
# ── Ensemble optimization ────────────────────────────────────────────
names = list(oofs.keys())
oof_mat = np.column_stack([rankdata(oofs[n]) for n in names])
test_mat = np.column_stack([rankdata(tests[n]) for n in names])

# 1) Equal-weight rank average baseline
eq_oof = oof_mat.mean(axis=1)
print(f'Equal rank-avg AUC:     {roc_auc_score(yv, eq_oof):.6f}')

# 2) Optuna weight optimization
def opt_weights(trial):
    w = np.array([trial.suggest_float(f'w_{n}', 0, 1) for n in names])
    w = w / (w.sum() + 1e-12)
    blend = (oof_mat * w).sum(axis=1)
    return roc_auc_score(yv, blend)

w_study = optuna.create_study(
    direction='maximize',
    sampler=optuna.samplers.TPESampler(seed=SEED),
)
w_study.optimize(opt_weights, n_trials=300, show_progress_bar=False)

bw = np.array([w_study.best_params[f'w_{n}'] for n in names])
bw = bw / bw.sum()
opt_oof = (oof_mat * bw).sum(axis=1)
opt_test = (test_mat * bw).sum(axis=1)
print(f'Optuna blend AUC:       {roc_auc_score(yv, opt_oof):.6f}')
print(f'  Weights: {dict(zip(names, bw.round(3)))}')

# 3) Ridge stacking meta-learner
oof_prob = np.column_stack([oofs[n] for n in names])
test_prob = np.column_stack([tests[n] for n in names])

skf_meta = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
meta_oof = np.zeros(len(yv))
meta_test = np.zeros(test_prob.shape[0])
for ti, vi in skf_meta.split(oof_prob, yv):
    meta = Ridge(alpha=1.0)
    meta.fit(oof_prob[ti], yv[ti])
    meta_oof[vi] = meta.predict(oof_prob[vi])
    meta_test += meta.predict(test_prob) / 5

print(f'Ridge stacking AUC:     {roc_auc_score(yv, meta_oof):.6f}')

# 4) Final = rank average of optuna blend + stacking
final_test = (rankdata(opt_test) + rankdata(meta_test)) / (2 * len(opt_test))
final_oof = (rankdata(opt_oof) + rankdata(meta_oof)) / (2 * len(opt_oof))
print(f'Final ensemble OOF AUC: {roc_auc_score(yv, final_oof):.6f}')

# Also try: best single submission (sometimes beats ensemble on public LB)
best_single_name = max(oofs, key=lambda n: roc_auc_score(yv, oofs[n]))
print(f'\nBest single model: {best_single_name} '
      f'(AUC={roc_auc_score(yv, oofs[best_single_name]):.6f})')

Equal rank-avg AUC:     0.689443
Optuna blend AUC:       0.690631
  Weights: {'lgbm_v2': np.float64(0.301), 'lgbm_v3': np.float64(0.235), 'lgbm_div': np.float64(0.051), 'xgb_v2': np.float64(0.06), 'xgb_v3': np.float64(0.038), 'catboost': np.float64(0.313), 'et': np.float64(0.002)}
Ridge stacking AUC:     0.690434
Final ensemble OOF AUC: 0.690775

Best single model: catboost (AUC=0.690080)


In [ ]:
# ── Generate submissions ─────────────────────────────────────────────
os.makedirs('submissions', exist_ok=True)

# Submission 1: Full ensemble
sub_ens = sample_sub.copy()
sub_ens[TARGET_COL] = final_test
sub_ens.to_csv('submissions/sub_ensemble.csv', index=False)

# Submission 2: Optuna blend only
sub_blend = sample_sub.copy()
sub_blend[TARGET_COL] = opt_test
sub_blend.to_csv('submissions/sub_optuna_blend.csv', index=False)

# Submission 3: Best single model
sub_single = sample_sub.copy()
sub_single[TARGET_COL] = tests[best_single_name]
sub_single.to_csv('submissions/sub_best_single.csv', index=False)

print('=== Submissions generated ===')
for f in ['sub_ensemble.csv', 'sub_optuna_blend.csv', 'sub_best_single.csv']:
    s = pd.read_csv(f'submissions/{f}')
    print(f'  {f:30s} mean={s[TARGET_COL].mean():.4f}  '
          f'min={s[TARGET_COL].min():.4f}  max={s[TARGET_COL].max():.4f}')

print('\n>>> Submit ALL THREE to the leaderboard and keep the best! <<<')
print('>>> Different submissions often score differently on public vs private LB <<<')

=== Submissions generated ===
  sub_ensemble.csv               mean=0.5000  min=0.0002  max=1.0000
  sub_optuna_blend.csv           mean=6489.0000  min=6.4990  max=12976.6674
  sub_best_single.csv            mean=0.2275  min=0.0358  max=0.7700

>>> Submit ALL THREE to the leaderboard and keep the best! <<<
>>> Different submissions often score differently on public vs private LB <<<
